In [1]:
import pandas as pd
import numpy as np
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
from sklearn.preprocessing import StandardScaler

In [8]:
# 1. 加载CSV
csv_path = "D:/yoyu/SA_Identification/Dataset/seawall/gwr_input_grid500m01.csv"
df = pd.read_csv(csv_path)

print(f"原始数据行数: {len(df)}")

print(f"原始数据行数: {len(df)}")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['Y_Prob', 'X1_Dist', 'X2_DEM', 'X3_NDVI', 'Center_X', 'Center_Y'], inplace=True)
print(f"清洗后数据行数: {len(df)}")

# 2. 准备坐标
coords = list(zip(df['Center_X'], df['Center_Y']))
coords = np.array(coords)

# 3. 准备 Y
y = df['Y_Prob'].values.reshape(-1, 1)

# ==========================================
# 【核心修复 1】：添加微小扰动防止奇异矩阵崩溃
# ==========================================
X_raw = df[['X1_Dist', 'X2_DEM', 'X3_NDVI']].values

# 给数据加上百万分之一的随机扰动 (Jittering)
# 这样即使一片滩涂的高程全是 0.0，也会变成 0.00001, -0.00002，避免了局部方差为 0
np.random.seed(42) # 保证每次运行结果一致
jitter = np.random.normal(0, 1e-5, X_raw.shape)
X_raw_jittered = X_raw + jitter

# 标准化
scaler = StandardScaler()
X = scaler.fit_transform(X_raw_jittered)

# 强制替换任何漏网的 NaN
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

print("数据准备就绪，开始搜索带宽...")

try:
    # 4. 带宽搜索
    selector = Sel_BW(coords, y, X, kernel='bisquare', spherical=False)
    bw = selector.search()
    print(f"最优带宽: {bw}")

    # 5. 运行 GWR
    model = GWR(coords, y, X, bw=bw, kernel='bisquare')
    results = model.fit()

    # ==========================================
    # 【核心修复 2】：强力清洗 Local R2，防止归一化崩塌
    # ==========================================
    local_R2 = results.localR2.flatten()
    
    # 将所有的 NaN 或 inf 替换为 0 (拟合最差)
    local_R2 = np.nan_to_num(local_R2, nan=0.0, posinf=1.0, neginf=0.0)
    
    # 地学意义：R2 应该在 [0, 1] 之间。
    # < 0 说明模型在这个局部完全失效（不确定性极大），我们强行设为 0
    # > 1 说明计算溢出，强行设为 1
    local_R2 = np.clip(local_R2, 0.0, 1.0)
    
    df['local_R2'] = local_R2

    # 6. 计算不确定性权重
    # R2 越小，不确定性越高
    uncertainty_weight = 1.0 - local_R2
    
    # 因为 R2 已经被 clip 到了 [0,1]，所以 uncertainty_weight 自然也在 [0,1]
    # 我们不需要再做 (x-min)/(max-min) 的归一化了！这避免了全变 0 的惨剧。
    df['uncertainty_weight'] = uncertainty_weight

    df.to_csv("gwr_results_with_weights.csv", index=False)
    print("\n[成功] GWR 运行完毕！")
    print("\n--- 最终权重统计分布 ---")
    print(df['uncertainty_weight'].describe())

except Exception as e:
    print(f"\n[运行出错] 错误信息: {e}")

原始数据行数: 10272
原始数据行数: 10272
清洗后数据行数: 10272
数据准备就绪，开始搜索带宽...
最优带宽: 49.0

[成功] GWR 运行完毕！

--- 最终权重统计分布 ---
count    10272.000000
mean         0.843507
std          0.283035
min          0.030166
25%          0.799347
50%          1.000000
75%          1.000000
max          1.000000
Name: uncertainty_weight, dtype: float64


d:\develop\miniconda3\envs\SA\Lib\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS
d:\develop\miniconda3\envs\SA\Lib\site-packages\mgwr\gwr.py:775: RuntimeWarning: invalid value encountered in divide
  return (self.TSS - self.RSS) / self.TSS


In [2]:
"""
GWDA Cell 1：用地理加权判别分析替换原来的 GWR
─────────────────────────────────────────────────────────────────────────────
修改原因：
  原方法对二值响应变量 Y_Prob（互花米草有/无）使用 GWR（连续型回归），
  违反了 GWR 的线性假设，localR2 的生态意义也不明确。
  改用 GWDA（Brunsdon et al., 2007），专为分类响应变量设计，
  输出每个位置的后验概率 P(Spartina=1|X, location)，
  值域天然在 [0,1]，直接作为空间先验图输入模型。

输出：
  gwr_results_with_weights.csv（列名与原来相同，uncertainty_weight 列
  现在是 GWDA 后验概率，而不是 1-localR2）
─────────────────────────────────────────────────────────────────────────────
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# ══════════════════════════════════════════════════════════════════════════
#  配置
# ══════════════════════════════════════════════════════════════════════════
CSV_PATH    = "D:/yoyu/SA_Identification/Dataset/seawall/gwr_input_grid500m01.csv"
OUTPUT_CSV  = "gwr_results_with_weights.csv"

# GWDA 核函数参数
# bandwidth：自适应带宽，即每个位置使用最近 k 个样本点
# 初始值设 50，如果样本点较少可以调小到 30
BANDWIDTH   = 50

# 响应变量列名（二值：有治理=1，无治理=0）
Y_COL       = "Y_Prob"

# 预测变量列名（与原来保持一致）
X_COLS      = ["X1_Dist", "X2_DEM", "X3_NDVI"]

# 坐标列名
COORD_X     = "Center_X"
COORD_Y     = "Center_Y"
# ══════════════════════════════════════════════════════════════════════════


def gaussian_kernel(distances: np.ndarray, bandwidth: float) -> np.ndarray:
    """高斯核权重，距离越近权重越大"""
    return np.exp(-0.5 * (distances / bandwidth) ** 2)


def adaptive_bisquare_kernel(distances: np.ndarray, k: int) -> np.ndarray:
    """
    自适应双平方核：取最近 k 个样本点，
    带宽 = 第 k 个最近邻的距离
    """
    sorted_d = np.sort(distances)
    bw = sorted_d[min(k, len(sorted_d) - 1)]
    if bw == 0:
        bw = 1e-6
    u = distances / bw
    w = np.where(u < 1, (1 - u ** 2) ** 2, 0.0)
    return w


def gwda_posterior(coords: np.ndarray, X: np.ndarray,
                   y: np.ndarray, bandwidth: int) -> np.ndarray:
    """
    地理加权判别分析（GWDA）
    在每个样本点位置，用自适应双平方核加权计算局部 LDA，
    返回该点属于 class=1（互花米草存在）的后验概率。

    Args:
        coords:    [N, 2] 坐标数组
        X:         [N, p] 标准化后的预测变量
        y:         [N]   二值标签（0/1）
        bandwidth: 自适应带宽（最近邻数量 k）
    Returns:
        posterior_prob: [N] 每个点的后验概率 P(y=1|X, location)
    """
    N = len(y)
    posterior_prob = np.zeros(N, dtype=np.float32)

    # 预先计算所有点对之间的欧氏距离矩阵
    print("  计算距离矩阵...")
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]  # [N, N, 2]
    dist_matrix = np.sqrt((diff ** 2).sum(axis=2))               # [N, N]

    print(f"  开始 GWDA 循环（{N} 个样本点，带宽 k={bandwidth}）...")
    for i in range(N):
        if i % 500 == 0:
            print(f"    进度: {i}/{N} ({i/N*100:.1f}%)")

        distances = dist_matrix[i]

        # 计算核权重
        weights = adaptive_bisquare_kernel(distances, bandwidth)

        # 排除权重为 0 的点（远距离点）
        valid = weights > 1e-8
        if valid.sum() < 4:
            # 有效样本太少，用全局先验概率代替
            posterior_prob[i] = y.mean()
            continue

        X_local = X[valid]
        y_local = y[valid]
        w_local = weights[valid]

        # 检查局部样本是否包含两个类别
        if len(np.unique(y_local)) < 2:
            # 局部只有一个类别，直接用局部类别比例
            posterior_prob[i] = float(y_local.mean())
            continue

        # 加权 LDA：用样本权重 sqrt(w) 重加权后做 LDA
        # sklearn LDA 不直接支持 sample_weight，用样本复制近似
        # 方法：将权重归一化后乘以一个放大系数作为 sample_weight
        w_norm = w_local / w_local.sum()

        try:
            lda = LinearDiscriminantAnalysis(solver='lsqr',
                                             shrinkage='auto')
            lda.fit(X_local, y_local, sample_weight=w_norm)
            # 获取 class=1 的后验概率
            prob = lda.predict_proba(X[i:i+1])[0]
            class_idx = list(lda.classes_).index(1) if 1 in lda.classes_ else -1
            if class_idx >= 0:
                posterior_prob[i] = float(prob[class_idx])
            else:
                posterior_prob[i] = 0.0
        except Exception:
            # 局部矩阵奇异，用局部均值代替
            posterior_prob[i] = float((y_local * w_norm).sum())

    return posterior_prob


# ── 主流程 ────────────────────────────────────────────────────────────────
print("1. 加载数据...")
df = pd.read_csv(CSV_PATH)
print(f"   原始数据行数: {len(df)}")

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=[Y_COL] + X_COLS + [COORD_X, COORD_Y], inplace=True)
print(f"   清洗后行数: {len(df)}")

# 二值化响应变量（Y_Prob > 0 视为有治理事件）
y = (df[Y_COL].values > 0).astype(int)
print(f"   正样本（有治理）: {y.sum()}  负样本（无治理）: {(y==0).sum()}")

# 坐标
coords = df[[COORD_X, COORD_Y]].values.astype(np.float64)

# 标准化预测变量（加微小抖动防奇异）
np.random.seed(42)
X_raw = df[X_COLS].values.astype(np.float64)
X_raw += np.random.normal(0, 1e-5, X_raw.shape)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\n2. 运行 GWDA（带宽 k={BANDWIDTH}）...")
posterior_prob = gwda_posterior(coords, X, y, BANDWIDTH)

# 统计
print(f"\n   后验概率统计:")
print(f"   min={posterior_prob.min():.4f}  max={posterior_prob.max():.4f}")
print(f"   mean={posterior_prob.mean():.4f}  std={posterior_prob.std():.4f}")

# 保存结果（列名与原来保持一致，Cell 2-4 无需修改）
df['gwda_posterior'] = posterior_prob
df['uncertainty_weight'] = posterior_prob   # 保持与下游 Cell 一致的列名

df.to_csv(OUTPUT_CSV, index=False)
print(f"\n3. 结果已保存: {OUTPUT_CSV}")
print("   列 'uncertainty_weight' = GWDA 后验概率 P(Spartina=1|X, location)")
print("   值域 [0,1]，直接作为空间先验图")
print("\n   后续 Cell 2-4（栅格化）和 Cell 5-7（切片）无需任何修改，直接运行即可。")

1. 加载数据...
   原始数据行数: 10272
   清洗后行数: 10272
   正样本（有治理）: 1150  负样本（无治理）: 9122

2. 运行 GWDA（带宽 k=50）...
  计算距离矩阵...
  开始 GWDA 循环（10272 个样本点，带宽 k=50）...
    进度: 0/10272 (0.0%)
    进度: 500/10272 (4.9%)
    进度: 1000/10272 (9.7%)
    进度: 1500/10272 (14.6%)
    进度: 2000/10272 (19.5%)
    进度: 2500/10272 (24.3%)
    进度: 3000/10272 (29.2%)
    进度: 3500/10272 (34.1%)
    进度: 4000/10272 (38.9%)
    进度: 4500/10272 (43.8%)
    进度: 5000/10272 (48.7%)
    进度: 5500/10272 (53.5%)
    进度: 6000/10272 (58.4%)
    进度: 6500/10272 (63.3%)
    进度: 7000/10272 (68.1%)
    进度: 7500/10272 (73.0%)
    进度: 8000/10272 (77.9%)
    进度: 8500/10272 (82.7%)
    进度: 9000/10272 (87.6%)
    进度: 9500/10272 (92.5%)
    进度: 10000/10272 (97.4%)

   后验概率统计:
   min=0.0000  max=0.9998
   mean=0.1090  std=0.2299

3. 结果已保存: gwr_results_with_weights.csv
   列 'uncertainty_weight' = GWDA 后验概率 P(Spartina=1|X, location)
   值域 [0,1]，直接作为空间先验图

   后续 Cell 2-4（栅格化）和 Cell 5-7（切片）无需任何修改，直接运行即可。


In [3]:
import pandas as pd
import rasterio
from rasterio.features import rasterize
from shapely.geometry import box
import numpy as np

# ==================== 配置区 ====================
CSV_PATH = "D:/yoyu/SA_Identification/project/gwr_results_with_weights.csv"

# 找一个完美的参考影像（用来获取全图的宽、高、坐标系和仿射变换矩阵）
# 建议用你最初的那个距离图或者 DEM 图，它们是完美的底图基准
REFERENCE_TIFF = "D:/yoyu/SA_Identification/Dataset/seawall/distance_to_coastline.tif" 

# 输出的空间先验图路径
OUTPUT_PRIOR_TIFF = "D:/yoyu/SA_Identification/Dataset/seawall/spatial_prior.tif"

GRID_SIZE = 500 # 必须与你提取网格时的大小一致
# ================================================

def csv_to_spatial_prior_tiff(csv_path, ref_tiff, out_tiff, grid_size):
    
    df = pd.read_csv(csv_path)

    with rasterio.open(ref_tiff) as src:
        meta = src.meta.copy()
        transform = src.transform
        width = src.width
        height = src.height
        
    # 把中心点坐标还原成 500x500 的方块多边形，并附带权重值
    shapes = []
    half_grid = grid_size / 2.0
    for _, row in df.iterrows():
        cx = row['Center_X']
        cy = row['Center_Y']
        weight = row['uncertainty_weight']
        
        # 确保权重是合理的浮点数
        if pd.isna(weight):
            weight = 0.0
            
        # 生成左下到右上的 Bounding Box
        geom = box(cx - half_grid, cy - half_grid, cx + half_grid, cy + half_grid)
        
        # rasterize 需要的格式: (geometry, value)
        shapes.append((geom, weight))

    print(f"4. 正在栅格化烧录 (Rasterizing) 到 {width}x{height} 的画布上...")
    # fill=0 极其重要：那些没有互花米草的海水/陆地背景，权重一律设为 0
    # 这样深度学习模型在扫描到海洋时，就不会被施加额外的注意力
    prior_array = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0.0,  
        dtype='float32'
    )

    print("5. 正在导出空间先验权重图...")
    meta.update(
        dtype=rasterio.float32,
        count=1,
        nodata=0.0, # 背景为 0
        compress='lzw'
    )

    with rasterio.open(out_tiff, 'w', **meta) as dst:
        dst.write(prior_array, 1)

    print(f"\n[大功告成] 完美的空间先验图已生成: {out_tiff}")
    print("你可以把它拖进 QGIS 里，用伪彩色（如热力图配色）查看不确定性的空间分布情况。")

if __name__ == "__main__":
    csv_to_spatial_prior_tiff(CSV_PATH, REFERENCE_TIFF, OUTPUT_PRIOR_TIFF, GRID_SIZE)

4. 正在栅格化烧录 (Rasterizing) 到 25091x38088 的画布上...
5. 正在导出空间先验权重图...

[大功告成] 完美的空间先验图已生成: D:/yoyu/SA_Identification/Dataset/seawall/spatial_prior.tif
你可以把它拖进 QGIS 里，用伪彩色（如热力图配色）查看不确定性的空间分布情况。


In [ ]:
# ==================== 配置区 ====================
CSV_PATH = "D:/yoyu/SA_Identification/project/gwr_results_with_weights.csv"
REFERENCE_TIFF = "D:/yoyu/SA_Identification/Dataset/seawall/distance_to_coastline.tif" 
OUTPUT_PRIOR_TIFF = "D:/yoyu/SA_Identification/Dataset/seawall/spatial_prior03.tif"
GRID_SIZE = 500
def csv_to_spatial_prior_tiff(csv_path, ref_tiff, out_tiff, grid_size):
    print("1. 读取 GWR 结果与执行【绝对事件掩膜】...")
    df = pd.read_csv(csv_path)
    
    # 【终极核心逻辑】：绝对事件掩膜 (Absolute Event Masking)
    # 只要这个网格内没有任何互花米草治理的痕迹（Y_Prob 严格等于 0.0）
    # 无论它是泥滩、长了海藻的水面、还是填充了高程 0 的空白区
    # 统统认定为“背景统计噪音”，其高权重必须被强制清零！
    # （因为模型在这些地方不需要寻找治理边界，只需要预测背景即可）
    
    # 使用极小的阈值防止浮点数精度问题 (比如 0.0001)
    mask_absolute_background = (df['Y_Prob'] < 0.0001)
    
    invalid_count = mask_absolute_background.sum()
    
    # 强制将这些无事件区域的注意力权重熄灭为 0
    df.loc[mask_absolute_background, 'uncertainty_weight'] = 0.0
    
    print(f"   - 强力封杀：已成功拦截 {invalid_count} 个完全无治理事件的背景网格。")
    print("   - 现在的权重图只会在‘发生了治理、且规律复杂’的地方亮起！")

    print(f"2. 读取参考模板影像: {ref_tiff}")
    with rasterio.open(ref_tiff) as src:
        meta = src.meta.copy()
        transform = src.transform
        width = src.width
        height = src.height
        
    print("3. 构建空间几何体 (Polygons)...")
    shapes = []
    half_grid = grid_size / 2.0
    for _, row in df.iterrows():
        cx = row['Center_X']
        cy = row['Center_Y']
        weight = row['uncertainty_weight']
        
        # 安全处理
        if pd.isna(weight):
            weight = 0.0
            
        geom = box(cx - half_grid, cy - half_grid, cx + half_grid, cy + half_grid)
        shapes.append((geom, weight))

    print(f"4. 正在栅格化烧录到 {width}x{height} 的全图上...")
    prior_array = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0.0,  
        dtype='float32'
    )

    print("5. 导出完美的空间先验图...")
    meta.update(
        dtype=rasterio.float32, 
        count=1, 
        nodata=0.0, 
        compress='lzw'
    )

    with rasterio.open(out_tiff, 'w', **meta) as dst:
        dst.write(prior_array, 1)

    print(f"\n[大功告成] 所有光滩噪音已被彻底清除！新图已生成: {out_tiff}")

if __name__ == "__main__":
    csv_to_spatial_prior_tiff(CSV_PATH, REFERENCE_TIFF, OUTPUT_PRIOR_TIFF, GRID_SIZE)

1. 读取 GWR 结果与执行【绝对事件掩膜】...
   - 强力封杀：已成功拦截 9122 个完全无治理事件的背景网格。
   - 现在的权重图只会在‘发生了治理、且规律复杂’的地方亮起！
2. 读取参考模板影像: D:/yoyu/SA_Identification/Dataset/seawall/distance_to_coastline.tif
3. 构建空间几何体 (Polygons)...
4. 正在栅格化烧录到 25091x38088 的全图上...
5. 导出完美的空间先验图...

[大功告成] 所有光滩噪音已被彻底清除！新图已生成: D:/yoyu/SA_Identification/Dataset/seawall/spatial_prior03.tif


In [4]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [5]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm import tqdm
import glob

In [6]:
# ================= 1. 基础配置 =================
# 原始大图的目录 (包含各个河口的文件夹)
DATA_ROOT = r"D:/yoyu/SA_Identification/Dataset/Estuaries"

# 已经切好的数据集目录 (里面有 A, B, label)
PATCH_DIR = r"D:/yoyu/SA_Identification/dataset_patches_2020_2024"

# 【新增】：我们刚刚生成的全景空间先验图路径
GLOBAL_PRIOR_TIFF = r"D:/yoyu/SA_Identification/Dataset/seawall/spatial_prior.tif"

PATCH_SIZE = 256

# ================= 2. 核心代码 =================
def create_prior_folder():
    # 为了不覆盖你之前的 weight，我们建一个新文件夹叫 spatial_prior
    prior_dir = os.path.join(PATCH_DIR, 'spatial_prior')
    os.makedirs(prior_dir, exist_ok=True)
    return prior_dir

def generate_aligned_priors():
    prior_dir = create_prior_folder()
    label_dir = os.path.join(PATCH_DIR, 'label')
    
    # 1. 获取所有已经存在的 label 切片文件名
    existing_labels = glob.glob(os.path.join(label_dir, "*.npy"))
    if not existing_labels:
        print(f" 错误：在 {label_dir} 中没找到任何 .npy 文件！")
        return
        
    print(f"在已有数据集中找到 {len(existing_labels)} 个切片。")
    
    # 2. 将文件名按河口 (estuary_name) 进行分组
    estuary_dict = {}
    for filepath in existing_labels:
        filename = os.path.basename(filepath)
        # 解析文件名: estuaryName_x_y.npy
        parts = filename.replace('.npy', '').split('_')
        x, y = int(parts[-2]), int(parts[-1])
        estuary_name = "_".join(parts[:-2]) 
        
        if estuary_name not in estuary_dict:
            estuary_dict[estuary_name] = []
        estuary_dict[estuary_name].append((filename, x, y))

    print(f"解析出 {len(estuary_dict)} 个河口区域需要处理。\n")

    # 3. 按河口逐个提取空间先验，并进行裁剪
    for estuary_name, patch_list in estuary_dict.items():
        print(f"[{estuary_name}] 正在对齐全局空间先验图...")
        estuary_folder = os.path.join(DATA_ROOT, estuary_name)
        
        # 寻找对应的 T1.tif 获取局部的范围和坐标系
        t1_files = glob.glob(os.path.join(estuary_folder, "*2020*.tif"))
        if not t1_files:
            print(f"   [跳过] 找不到 {estuary_name} 的 T1.tif！")
            continue
            
        t1_path = t1_files[0]
        
        # === A. 获取当前河口影像的元数据 ===
        with rasterio.open(t1_path) as src_t1:
            t1_transform = src_t1.transform
            t1_crs = src_t1.crs
            t1_height = src_t1.height
            t1_width = src_t1.width
            
        # === B. 从全局 Prior 中抠出与当前河口完全对齐的局部 Prior ===
        prior_full = np.zeros((t1_height, t1_width), dtype=np.float32)
        
        with rasterio.open(GLOBAL_PRIOR_TIFF) as src_prior:
            # 使用 reproject 强行将全局图“塞入”局部图的网格中，保证像素 100% 对齐
            reproject(
                source=rasterio.band(src_prior, 1),
                destination=prior_full,
                src_transform=src_prior.transform,
                src_crs=src_prior.crs,
                dst_transform=t1_transform,
                dst_crs=t1_crs,
                resampling=Resampling.nearest # 由于原始网格是 500m，用最近邻保持权重数值不被模糊
            )
            
        # === C. 根据已有文件的 x, y 坐标裁剪权重图 ===
        print(f"   正在裁剪并保存 {len(patch_list)} 个先验切片...")
        for filename, x, y in tqdm(patch_list, desc="Cropping Priors", leave=False):
            # 切出 256x256 的权重块
            patch_prior = prior_full[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            
            # 保存到 spatial_prior 文件夹
            out_path = os.path.join(prior_dir, filename)
            np.save(out_path, patch_prior)
            
        print(f"   {estuary_name} 先验切片生成完毕！\n")

    print(f"\n[大功告成] 所有空间先验切片全部生成！存放于: {prior_dir}")

if __name__ == "__main__":
    generate_aligned_priors()

在已有数据集中找到 492 个切片。
解析出 1 个河口区域需要处理。

[EstuariesA] 正在对齐全局空间先验图...
   正在裁剪并保存 492 个先验切片...


   EstuariesA 先验切片生成完毕！


[大功告成] 所有空间先验切片全部生成！存放于: D:/yoyu/SA_Identification/dataset_patches_2020_2024\spatial_prior
